## IMPORTS


In [3]:
import pandas as pd
import numpy as np
import optuna
from sklearn.impute import SimpleImputer
import os
from imblearn.pipeline import Pipeline 
import xgboost as  xgb
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score, precision_score
import sys
from sklearn.experimental import enable_iterative_imputer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from imblearn.under_sampling import RandomUnderSampler, TomekLinks
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.combine import SMOTETomek, SMOTEENN
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import joblib
import logging
from sklearn.model_selection import train_test_split
import optuna
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.svm import SVC
import lightgbm as lgb
from sklearn.utils.class_weight import compute_sample_weight
import warnings
warnings.filterwarnings("ignore")




ruta_raiz = os.path.abspath('..')
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

%load_ext autoreload
%autoreload 2
from Data_preprocesing.IQRCapper import IQRCapper



c:\Users\gonzalo.iglesias\AppData\Local\miniconda3\envs\machinelearning2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOS OUTPUTS DE OPTUNA QUE LOS IMPRIMA EN UN TXT PARA SIN VOLVERLO A CORRER

In [4]:
log_file = logging.FileHandler("optuna_resultados.txt")
log_file.setFormatter(logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s'))
optuna.logging.get_logger("optuna").addHandler(log_file)

In [ ]:
TRAIN = False
SAVE_MODEL = False
CHARGE_XBOOST= False
CHOOSE_SAMPLER = False

## LOAD DATA + SAVE FUCTION


In [5]:
X_train = pd.read_csv("../Train_test_split/X_train.csv")
y_train = pd.read_csv("../Train_test_split/y_train.csv")
X_test = pd.read_csv("../Train_test_split/X_test.csv")
y_test = pd.read_csv("../Train_test_split/y_test.csv")



target_folder = '../comparations'

def save_experiment(model_name, imbalance_method, accuracy, precision, recall, f1, roc_auc):
    new_result = {
        'Model': [model_name],
        'Imbalance_Method': [imbalance_method],
        'Accuracy': [round(accuracy, 4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall, 4)],
        'F1_Score': [round(f1, 4)],
        'ROC_AUC': [round(roc_auc, 4)]
    }
    df_new = pd.DataFrame(new_result)
    
    # Point the file to inside the folder
    csv_file = f'{target_folder}/imbalance_results.csv'
    
    if os.path.exists(csv_file):
        df_new.to_csv(csv_file, mode='a', header=False, index=False)
    else:
        df_new.to_csv(csv_file, index=False)
        
    print(f"✅ Success! Results for {model_name} + {imbalance_method} saved in the '{target_folder}' folder.")


## XGBOOST

In [6]:
num_classes = int(y_train['customer_segment'].nunique())
modelo_xgb = xgb.XGBClassifier(
    objective='multi:softmax', 
    num_class=num_classes, 
    random_state=42,
    eval_metric='mlogloss'
)

## DEFINIMOS LOS RANGOS Y CATEGORIAS QUE QUEREMOS QUE OPTUNA OPTIMICE

In [42]:
if CHOOSE_SAMPLER:
    imbalance_methods = {
        "Baseline": None,
        "RandomUnderSampler":     RandomUnderSampler(random_state=42),
        "TomekLinks":             TomekLinks(),
        "RandomOverSampler":      RandomOverSampler(random_state=42),
        "SMOTE":                  SMOTE(random_state=42),
        "ADASYN":                 ADASYN(random_state=42),
        "SMOTETomek":             SMOTETomek(random_state=42),
        "SMOTEENN":               SMOTEENN(random_state=42),
    }


    for method_name, sampler in imbalance_methods.items():
        print(f"\n--- {method_name} ---")

        

        if sampler is None:
            pipe = Pipeline([
                ('capping_outliers', IQRCapper(factor=1.5)),
                ('imputacion', SimpleImputer(strategy='mean')),
                ('scaler', StandardScaler()),
                ('classifier', modelo_xgb)])
        else:
            pipe = Pipeline([
                ('capping_outliers', IQRCapper(factor=1.5)),
                ('imputacion', SimpleImputer(strategy='mean')),
                ('scaler', StandardScaler()),
                ('sampler', sampler), 
                ('classifier', modelo_xgb)])

        pipe.fit(X_train, y_train)

        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)

        accuracy  = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

        print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

        save_experiment(
            model_name="XgBoost",
            imbalance_method=method_name,
            accuracy=accuracy,
            precision=precision,
            recall=recall,
            f1=f1,
            roc_auc=roc_auc
        )


--- Baseline ---
Accuracy: 0.7531 | F1: 0.7517 | ROC-AUC: 0.9009
✅ Success! Results for XgBoost + Baseline saved in the '../comparations' folder.

--- RandomUnderSampler ---
Accuracy: 0.7276 | F1: 0.7313 | ROC-AUC: 0.8928
✅ Success! Results for XgBoost + RandomUnderSampler saved in the '../comparations' folder.

--- TomekLinks ---
Accuracy: 0.7564 | F1: 0.7545 | ROC-AUC: 0.9018
✅ Success! Results for XgBoost + TomekLinks saved in the '../comparations' folder.

--- RandomOverSampler ---
Accuracy: 0.7425 | F1: 0.7455 | ROC-AUC: 0.8988
✅ Success! Results for XgBoost + RandomOverSampler saved in the '../comparations' folder.

--- SMOTE ---
Accuracy: 0.7501 | F1: 0.7512 | ROC-AUC: 0.8989
✅ Success! Results for XgBoost + SMOTE saved in the '../comparations' folder.

--- ADASYN ---
Accuracy: 0.7442 | F1: 0.7448 | ROC-AUC: 0.8983
✅ Success! Results for XgBoost + ADASYN saved in the '../comparations' folder.

--- SMOTETomek ---
Accuracy: 0.7509 | F1: 0.7521 | ROC-AUC: 0.8992
✅ Success! Results

In [ ]:
def objective(trial):
    

    # ESCOGER TIPO DE MUESTREO
    nombre_metodo = "TomekLinks" 
    metodo_elegido = TomekLinks()
    # ESCOGER LOS HIPERPARÁMETROS DE XGBOOST
    param_n_estimators = trial.suggest_int('n_estimators', 50, 1000)
    param_max_depth = trial.suggest_int('max_depth', 3, 8)  
    param_learning_rate = trial.suggest_float('learning_rate', 0.005, 0.3, log=True)
    param_gamma = trial.suggest_float('gamma', 0.0, 5.0 )
    param_lambda = trial.suggest_float('lambda', 0.5, 10.0, log=True) 
    param_alpha = trial.suggest_float('alpha', 0.0, 5.0)
    param_subsample = trial.suggest_float('subsample',        0.5, 1.0)
    param_colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    param_min_child_weight = trial.suggest_int(  'min_child_weight', 1, 20)  # 
    param_max_delta_step = trial.suggest_float('max_delta_step', 0.0, 1.0)
    param_iqr_factor = trial.suggest_float('iqr_factor', 1.0, 3.0)



    modelo_xgb = xgb.XGBClassifier(
        n_estimators = param_n_estimators,
        max_depth = param_max_depth,
        learning_rate = param_learning_rate,
        gamma = param_gamma,
        reg_lambda = param_lambda,
        reg_alpha = param_alpha,
        subsample = param_subsample,
        colsample_bytree = param_colsample_bytree,
        min_child_weight = param_min_child_weight,
        max_delta_step = param_max_delta_step,
        objective = 'multi:softmax',
        eval_metric = 'mlogloss',
        num_class = len(np.unique(y_train)),
        random_state = 42,
        n_jobs = -1
    )

    pipeline = Pipeline([
        ('capping_outliers', IQRCapper(factor=param_iqr_factor)),
        ('imputacion',       SimpleImputer(strategy='mean')),
        ('scaler',           StandardScaler()),
        ('sampler', metodo_elegido),
        ('clf',              modelo_xgb)
    ])

    
    sample_weights = compute_sample_weight('balanced', y_train)


    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(
    pipeline,
    X_train,
    y_train,
    scoring = 'roc_auc_ovr_weighted',
    cv=cv,
    n_jobs=1)

    return np.mean(scores)



if TRAIN:
    best_value = [None]
    no_improve_count = [0]

    def early_stopping_callback(study, trial):
        if best_value[0] is None or study.best_value > best_value[0]:
            best_value[0] = study.best_value
            no_improve_count[0] = 0
        else:
            no_improve_count[0] += 1
        if no_improve_count[0] >= 50:
            study.stop()

    study = optuna.create_study(direction="maximize", 
                                sampler=optuna.samplers.TPESampler(seed=42),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)  
)
    study.optimize(objective, n_trials=200, callbacks=[early_stopping_callback], show_progress_bar = True)
    print(f"Mejor F1-Score Macro: {study.best_value:.4f}")
    print("Mejor combinación encontrada:")
    for key, value in study.best_params.items():
        print(f"  - {key}: {value}")

    with open("mejores_parametros_XGBoost.txt", "w") as archivo:
        archivo.write("Mejores hiperparámetros encontrados para XGBoost:\n")
        archivo.write("="*40 + "\n")
        for key, value in study.best_params.items():
            archivo.write(f"  - {key}: {value}\n")

    print("¡Parámetros guardados correctamente en 'mejores_parametros.txt'!")
    
            
  

GUARDAMOS EL MEJOR MODELO

In [ ]:
if TRAIN:
    print("Reconstruyendo y entrenando el modelo final con toda la data...")
    
    mejor_metodo_nombre = "TomekLinks"
    mejor_metodo = TomekLinks()
    sample_weights = compute_sample_weight('balanced', y_train)

    mejor_xgb = xgb.XGBClassifier(
        n_estimators=study.best_params['n_estimators'],
        max_depth=study.best_params['max_depth'],
        learning_rate=study.best_params['learning_rate'],
        subsample=study.best_params['subsample'],
        colsample_bytree=study.best_params['colsample_bytree'],
        objective = 'multi:softprob',
        eval_metric='mlogloss',
        tree_method = 'hist', 
        random_state=42,
        n_jobs=-1
    )

    pasos_finales = [
        ('capping_outliers', IQRCapper(factor=study.best_params["iqr_factor"])),
        ('imputacion', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
        ('sampler', mejor_metodo),
        ('clf', mejor_xgb)
    ]



    pipeline_final = Pipeline(pasos_finales)


if SAVE_MODEL:
    joblib.dump(pipeline_final, 'modelo_xgboost_ganador.pkl')
    pipeline_final.fit(X_train, y_train)
    if SAVE_MODEL:
        joblib.dump(pasos_finales, 'modelo_xgboost_ganador.pkl')
        print("¡Modelo final entrenado y guardado correctamente como 'modelo_xgboost_ganador.pkl'!")
        


In [ ]:
if CHARGE_XBOOST:
    pasos_cargados = joblib.load('modelo_xgboost_ganador.pkl')
    pipeline = Pipeline(pasos_cargados)

    X_train = pd.read_csv("../Train_test_split/X_train.csv")
    y_train = pd.read_csv("../Train_test_split/y_train.csv")
    X_test = pd.read_csv("../Train_test_split/X_test.csv")
    y_test = pd.read_csv("../Train_test_split/y_test.csv")

    y_pred       = pipeline.predict(X_test)
    y_pred_proba = pipeline.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba, 
                            multi_class='ovr',  
                            average='weighted')  

    new_result = {
        'Model': ["XGBoost"],
        'Imbalance_Method': [pasos_cargados[3][1]],
        'Accuracy': [round(accuracy, 4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall, 4)],
        'F1_Score': [round(f1, 4)],
        'ROC_AUC': [round(roc_auc, 4)]
    }

    print(new_result)



## STACKING CLASIFFIER

In [6]:
TRAINSTACK =True
CHOOSESAMPLER = False
CHARGE_STACKING = True
SAVE_MODEL_STACK = True

In [7]:
imbalance_methods = {
    "Baseline": None,
    "RandomUnderSampler":     RandomUnderSampler(random_state=42),
    "TomekLinks":             TomekLinks(),
    "RandomOverSampler":      RandomOverSampler(random_state=42),
    "SMOTE":                  SMOTE(random_state=42),
    "ADASYN":                 ADASYN(random_state=42),
    "SMOTETomek":             SMOTETomek(random_state=42),
    "SMOTEENN":               SMOTEENN(random_state=42),
}

# ============================================================
# FASE 1: Selección del mejor sampler con LightGBM (proxy rápido)
# ============================================================
if CHOOSESAMPLER:
    best_sampler_name = None
    best_f1 = 0
    quick_model = lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1)

    for method_name, sampler in imbalance_methods.items():
        steps = [
            ('capping_outliers', IQRCapper(factor=1.5)),
            ('imputacion', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
        ]
        if sampler is not None:
            steps.append(('sampler', sampler))
        steps.append(('classifier', quick_model))

        pipe = Pipeline(steps)
        pipe.fit(X_train, y_train)

        y_pred = pipe.predict(X_test)
        y_proba = pipe.predict_proba(X_test)

        accuracy  = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='weighted')
        recall = recall_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')
        roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

        print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

        save_experiment(
            model_name="Stacking Clafier",
            imbalance_method=method_name,
            accuracy=accuracy,
            precision=precision,
            recall=recall,
            f1=f1,
            roc_auc=roc_auc
        )
        print(f"{method_name}: F1={f1:.4f}")
        if method_name == "Baseline": f1+=0.01   #Vamos a dar una pequeña ventaja a lo simple
        if f1 > best_f1:
            best_f1          = f1
            best_sampler_name = method_name

    print(f"\nMejor sampler: {best_sampler_name} (F1={best_f1:.4f})") 
best_sampler_name = "Baseline"
# ============================================================
# FASE 2: Stacking final solo con el mejor sampler
# ============================================================
modelos_base = [
    ('xgb', xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        eval_metric='mlogloss',
        n_estimators=100,
        max_depth=4,
        tree_method='hist',
        random_state=42
    )),
    ('rf', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )),
    ('svm', CalibratedClassifierCV(
        LinearSVC(random_state=42, max_iter=2000),
        cv=3
    )),
    ('lgb', lgb.LGBMClassifier(random_state=42, n_jobs=-1))
]

stacking_clf = StackingClassifier(
    estimators=modelos_base,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=-1
)

best_sampler = imbalance_methods["Baseline"]

steps_final = [
    ('capping_outliers', IQRCapper(factor=1.5)),
    ('imputacion', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
]
if best_sampler is not None:
    steps_final.append(('sampler', best_sampler))
steps_final.append(('classifier', stacking_clf))

pipe_final = Pipeline(steps_final)
pipe_final.fit(X_train, y_train)

y_pred  = pipe_final.predict(X_test)
y_proba = pipe_final.predict_proba(X_test)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')
roc_auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')

print(f"\n--- Stacking Final ({best_sampler_name}) ---")
print(f"Accuracy: {accuracy:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")


"""
Entrenar el Stacking Classifier completo con cada uno de los 8 métodos de balanceo es computacionalmente inviable, 
ya que cada iteración implica reentrenar múltiples modelos costosos con validación cruzada. 
Para reducir el coste sin sacrificar rigor, se adopta una estrategia en dos fases: primero se selecciona el método de balanceo óptimo usando 
LightGBM como modelo proxy —rápido, robusto y habitualmente correlacionado en rendimiento con ensambles más complejos—, y 
posteriormente se entrena el Stacking únicamente con dicho método. Esto reduce el número de entrenamientos costosos de 8 a 1, manteniendo una selección informada del sampler.
"""


--- Stacking Final (Baseline) ---
Accuracy: 0.7640 | F1: 0.7622 | ROC-AUC: 0.9068


'\nEntrenar el Stacking Classifier completo con cada uno de los 8 métodos de balanceo es computacionalmente inviable, \nya que cada iteración implica reentrenar múltiples modelos costosos con validación cruzada. \nPara reducir el coste sin sacrificar rigor, se adopta una estrategia en dos fases: primero se selecciona el método de balanceo óptimo usando \nLightGBM como modelo proxy —rápido, robusto y habitualmente correlacionado en rendimiento con ensambles más complejos—, y \nposteriormente se entrena el Stacking únicamente con dicho método. Esto reduce el número de entrenamientos costosos de 8 a 1, manteniendo una selección informada del sampler.\n'

In [12]:
if TRAINSTACK:
    def objective_stacking(trial):

        X_opt, _, y_opt, _ = train_test_split(
            X_train, y_train,
            train_size=25000,
            stratify=y_train,
            random_state=42
        )
        # ── META-MODELO: LogisticRegression completa ──────────────────────────
        solver = trial.suggest_categorical('meta_solver', ['saga', 'lbfgs'])
        penalty = trial.suggest_categorical('meta_penalty', ['l2', 'elasticnet']) if solver == 'saga' \
                else 'l2'  

        meta_model = LogisticRegression(
            C          = trial.suggest_float('meta_C', 1e-3, 10.0, log=True),
            penalty    = penalty,
            solver     = solver,
            l1_ratio   = trial.suggest_float('meta_l1_ratio', 0.0, 1.0)
                        if penalty == 'elasticnet' else None,
            max_iter   = trial.suggest_int('meta_max_iter', 500, 3000),
            tol        = trial.suggest_float('meta_tol', 1e-5, 1e-2, log=True),
            class_weight = trial.suggest_categorical('meta_class_weight', [None, 'balanced']),
            random_state = 42
        )

        # ── BASE LEARNER 1: XGBoost ───────────────────────────────────────
        xgb_model = xgb.XGBClassifier(
            n_estimators = trial.suggest_int('xgb_n_estimators', 50, 500),
            max_depth = trial.suggest_int('xgb_max_depth',3, 8),
            learning_rate = trial.suggest_float('xgb_learning_rate', 0.005, 0.3, log=True),
            subsample = trial.suggest_float('xgb_subsample', 0.5, 1.0),
            colsample_bytree = trial.suggest_float('xgb_colsample_bytree',0.5, 1.0),
            gamma = trial.suggest_float('xgb_gamma', 0.0, 5.0),
            reg_lambda = trial.suggest_float('xgb_lambda', 0.5, 10.0, log=True),
            reg_alpha = trial.suggest_float('xgb_alpha', 0.0, 5.0),
            min_child_weight = trial.suggest_int('xgb_min_child_weight', 1, 20),
            objective = 'multi:softprob',
            eval_metric = 'mlogloss',
            num_class = len(np.unique(y_train)),
            tree_method = 'hist',
            random_state = 42,
            n_jobs = -1
        )

        # ── BASE LEARNER 2: Random Forest ────────────────────────────────
        rf_model = RandomForestClassifier(
            n_estimators = trial.suggest_int('rf_n_estimators', 50, 500),
            max_depth = trial.suggest_int('rf_max_depth', 3, 20),
            max_features = trial.suggest_float('rf_max_features', 0.3, 1.0),
            min_samples_split = trial.suggest_int('rf_min_samples_split', 2, 20),
            min_samples_leaf  = trial.suggest_int('rf_min_samples_leaf', 1, 10),
            random_state = 42,
            n_jobs = -1
        )

        # ── BASE LEARNER 3: SVM (rbf, con probabilidades) ────────────────
        svm_model = SVC(
            C = trial.suggest_float('svm_C', 1e-2, 100.0, log=True),
            gamma = trial.suggest_float('svm_gamma', 1e-4, 1.0, log=True),
            kernel = 'rbf',
            probability = True,
            random_state= 42
        )

        # ── BASE LEARNER 4: LightGBM ─────────────────────────────────────
        lgbm_model = lgb.LGBMClassifier(
            n_estimators  = trial.suggest_int('lgbm_n_estimators', 50, 500),
            max_depth = trial.suggest_int('lgbm_max_depth', 3, 8),
            learning_rate = trial.suggest_float('lgbm_learning_rate', 0.005, 0.3, log=True),
            num_leaves = trial.suggest_int('lgbm_num_leaves', 15, 127),
            subsample = trial.suggest_float('lgbm_subsample', 0.5, 1.0),
            colsample_bytree = trial.suggest_float('lgbm_colsample_bytree', 0.5, 1.0),
            reg_alpha = trial.suggest_float('lgbm_alpha', 0.0, 5.0),
            reg_lambda = trial.suggest_float('lgbm_lambda', 0.5, 10.0, log=True),
            objective = 'multiclass',
            random_state  = 42,
            n_jobs = -1,
            verbose = -1
        )

        # ── STACKING ─────────────────────────────────────────────────────
        stacking = StackingClassifier(
            estimators=[
                ('xgb',xgb_model),
                ('rf', rf_model),
                ('svm',svm_model),
                ('lgbm', lgbm_model),
            ],
            final_estimator = meta_model,
            cv = 3,
            stack_method = 'predict_proba',
            n_jobs = -1
        )

        # ── PIPELINE ─────────────────────────────────────────────────────
        steps = [
            ('capping_outliers', IQRCapper(factor=trial.suggest_float('iqr_factor', 1.0, 3.0))),
            ('imputacion', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler()),
        ]
        if best_sampler is not None:
            steps.append(('sampler', best_sampler))
        steps.append(('classifier', stacking))

        pipeline = Pipeline(steps)

        cv_splitter = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        sample_weights = compute_sample_weight('balanced', y_train)

        scores = cross_val_score(
            pipeline,
            X_opt, y_opt,       
            cv      = cv_splitter,
            scoring = 'roc_auc_ovr_weighted',
            n_jobs  = 1
        )

        return np.mean(scores)


# ── EJECUCIÓN ─────────────────────────────────────────────────────────
if TRAINSTACK:
    best_value = [None]
    no_improve_count = [0]

    
    study = optuna.create_study(
        direction = "maximize",
        sampler = optuna.samplers.TPESampler(seed=42),
        pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5)
    )
    def early_stopping_callback(study, trial):
        if best_value[0] is None or study.best_value > best_value[0]:
            best_value[0] = study.best_value
            no_improve_count[0] = 0
        else:
            no_improve_count[0] += 1
        if no_improve_count[0] >= 50:
            study.stop()


    study.optimize(
        objective_stacking,
        n_trials = 20,
        callbacks = [early_stopping_callback],
        show_progress_bar = True
    )

    print(f"\nMejor ROC-AUC (weighted OvR): {study.best_value:.4f}")
    print("Mejores hiperparámetros:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")

    with open("mejores_parametros_Stacking.txt", "w") as f:
        f.write("Mejores hiperparámetros — Stacking Classifier\n")
        f.write("=" * 45 + "\n")
        for k, v in study.best_params.items():
            f.write(f"  {k}: {v}\n")

    print("\n¡Parámetros guardados en 'mejores_parametros_Stacking.txt'!")

[I 2026-03-17 00:27:23,153] A new study created in memory with name: no-name-783e95e5-6475-473a-9c3f-4a7ae687033b
Best trial: 0. Best value: 0.904794:   5%|▌         | 1/20 [07:52<2:29:35, 472.41s/it]

[I 2026-03-17 00:35:15,566] Trial 0 finished with value: 0.9047942154027853 and parameters: {'meta_solver': 'lbfgs', 'meta_C': 0.8471801418819978, 'meta_max_iter': 1997, 'meta_tol': 2.9380279387035334e-05, 'meta_class_weight': None, 'xgb_n_estimators': 440, 'xgb_max_depth': 6, 'xgb_learning_rate': 0.09078835431980901, 'xgb_subsample': 0.5102922471479012, 'xgb_colsample_bytree': 0.9849549260809971, 'xgb_gamma': 4.162213204002109, 'xgb_lambda': 0.9445600138094694, 'xgb_alpha': 0.9091248360355031, 'xgb_min_child_weight': 4, 'rf_n_estimators': 187, 'rf_max_depth': 12, 'rf_max_features': 0.602361513049481, 'rf_min_samples_split': 7, 'rf_min_samples_leaf': 7, 'svm_C': 0.03613894271216528, 'svm_gamma': 0.0014742753159914669, 'lgbm_n_estimators': 215, 'lgbm_max_depth': 5, 'lgbm_learning_rate': 0.12448918446337819, 'lgbm_num_leaves': 37, 'lgbm_subsample': 0.7571172192068059, 'lgbm_colsample_bytree': 0.7962072844310213, 'lgbm_alpha': 0.23225206359998862, 'lgbm_lambda': 3.0860579740535368, 'iqr_f

Best trial: 0. Best value: 0.904794:  10%|█         | 2/20 [15:15<2:16:30, 455.04s/it]

[I 2026-03-17 00:42:38,441] Trial 1 finished with value: 0.901920310171024 and parameters: {'meta_solver': 'lbfgs', 'meta_C': 7.2866537374910445, 'meta_max_iter': 2521, 'meta_tol': 8.200518402245828e-05, 'meta_class_weight': 'balanced', 'xgb_n_estimators': 248, 'xgb_max_depth': 3, 'xgb_learning_rate': 0.03797252233376349, 'xgb_subsample': 0.5171942605576092, 'xgb_colsample_bytree': 0.954660201039391, 'xgb_gamma': 1.2938999080000846, 'xgb_lambda': 3.6385753170854684, 'xgb_alpha': 1.5585553804470549, 'xgb_min_child_weight': 11, 'rf_n_estimators': 296, 'rf_max_depth': 6, 'rf_max_features': 0.9787092394351908, 'rf_min_samples_split': 16, 'rf_min_samples_leaf': 10, 'svm_C': 37.95853142670641, 'svm_gamma': 0.024637685958997457, 'lgbm_n_estimators': 465, 'lgbm_max_depth': 3, 'lgbm_learning_rate': 0.011154681546966926, 'lgbm_num_leaves': 20, 'lgbm_subsample': 0.6626651653816322, 'lgbm_colsample_bytree': 0.6943386448447411, 'lgbm_alpha': 1.3567451588694794, 'lgbm_lambda': 5.986629236379358, 'iq

Best trial: 0. Best value: 0.904794:  15%|█▌        | 3/20 [21:26<1:58:03, 416.66s/it]

...

[I 2026-03-17 02:46:55,103] Trial 17 finished with value: 0.9027929172850052 and parameters: {'meta_solver': 'lbfgs', 'meta_C': 0.022340908638417582, 'meta_max_iter': 1667, 'meta_tol': 2.8533620837790732e-05, 'meta_class_weight': None, 'xgb_n_estimators': 297, 'xgb_max_depth': 6, 'xgb_learning_rate': 0.025707538934836998, 'xgb_subsample': 0.6062914942092245, 'xgb_colsample_bytree': 0.9158873213786554, 'xgb_gamma': 4.959782326816241, 'xgb_lambda': 0.8635599491758744, 'xgb_alpha': 0.6958396728807502, 'xgb_min_child_weight': 9, 'rf_n_estimators': 61, 'rf_max_depth': 4, 'rf_max_features': 0.7563996497112966, 'rf_min_samples_split': 4, 'rf_min_samples_leaf': 8, 'svm_C': 0.05137096719989351, 'svm_gamma': 0.006310253140735259, 'lgbm_n_estimators': 147, 'lgbm_max_depth': 4, 'lgbm_learning_rate': 0.2479474446858831, 'lgbm_num_leaves': 77, 'lgbm_subsample': 0.9899226356996808, 'lgbm_colsample_bytree': 0.62148645855588, 'lgbm_alpha': 2.133456145286303, 'lgbm_lambda': 3.3578850648170966, 'iqr_fact

Best trial: 0. Best value: 0.904794:  95%|█████████▌| 19/20 [2:24:25<05:37, 337.46s/it]

[I 2026-03-17 02:51:48,958] Trial 18 finished with value: 0.9036541205487182 and parameters: {'meta_solver': 'lbfgs', 'meta_C': 0.008678195662833635, 'meta_max_iter': 1763, 'meta_tol': 0.00013094143634070684, 'meta_class_weight': None, 'xgb_n_estimators': 289, 'xgb_max_depth': 4, 'xgb_learning_rate': 0.08201121026130048, 'xgb_subsample': 0.6979435315858737, 'xgb_colsample_bytree': 0.9909270315731906, 'xgb_gamma': 4.354166705372391, 'xgb_lambda': 2.045823755563875, 'xgb_alpha': 1.3397903908341693, 'xgb_min_child_weight': 11, 'rf_n_estimators': 98, 'rf_max_depth': 20, 'rf_max_features': 0.8710157634739196, 'rf_min_samples_split': 11, 'rf_min_samples_leaf': 9, 'svm_C': 3.7222717245645813, 'svm_gamma': 0.0013841533145714248, 'lgbm_n_estimators': 95, 'lgbm_max_depth': 6, 'lgbm_learning_rate': 0.17673937741277923, 'lgbm_num_leaves': 18, 'lgbm_subsample': 0.9869658603544285, 'lgbm_colsample_bytree': 0.5446767938921808, 'lgbm_alpha': 3.443306009409624, 'lgbm_lambda': 8.844288907321529, 'iqr_fa

Best trial: 0. Best value: 0.904794: 100%|██████████| 20/20 [2:29:30<00:00, 448.54s/it]

[I 2026-03-17 02:56:54,001] Trial 19 finished with value: 0.9046117632775208 and parameters: {'meta_solver': 'lbfgs', 'meta_C': 0.07207008258245226, 'meta_max_iter': 1584, 'meta_tol': 0.00016812389436365578, 'meta_class_weight': None, 'xgb_n_estimators': 428, 'xgb_max_depth': 4, 'xgb_learning_rate': 0.19374417403964994, 'xgb_subsample': 0.6115241529415376, 'xgb_colsample_bytree': 0.9179301674644511, 'xgb_gamma': 3.416449763813842, 'xgb_lambda': 0.7662213216417383, 'xgb_alpha': 0.581569764539638, 'xgb_min_child_weight': 15, 'rf_n_estimators': 300, 'rf_max_depth': 9, 'rf_max_features': 0.7529611021135593, 'rf_min_samples_split': 10, 'rf_min_samples_leaf': 7, 'svm_C': 0.4196374381805806, 'svm_gamma': 0.010097996115732158, 'lgbm_n_estimators': 125, 'lgbm_max_depth': 4, 'lgbm_learning_rate': 0.2568415667679073, 'lgbm_num_leaves': 43, 'lgbm_subsample': 0.9115170861868737, 'lgbm_colsample_bytree': 0.6118302780841811, 'lgbm_alpha': 1.8751956772293397, 'lgbm_lambda': 0.8814013606087064, 'iqr_fa

In [13]:
if SAVE_MODEL_STACK:
    print("Reconstruyendo y entrenando el modelo final con toda la data...")

    best = study.best_params

    # ── META-MODELO ──────────────────────────────────────────────────
    meta_model = LogisticRegression(C=best['meta_C'], max_iter=1000, random_state=42)

    # ── BASE LEARNER 1: XGBoost ──────────────────────────────────────
    xgb_model = xgb.XGBClassifier(
        n_estimators = best['xgb_n_estimators'],
        max_depth = best['xgb_max_depth'],
        learning_rate = best['xgb_learning_rate'],
        subsample = best['xgb_subsample'],
        colsample_bytree = best['xgb_colsample_bytree'],
        gamma = best['xgb_gamma'],
        reg_lambda = best['xgb_lambda'],
        reg_alpha = best['xgb_alpha'],
        min_child_weight = best['xgb_min_child_weight'],
        objective = 'multi:softprob',
        eval_metric = 'mlogloss',
        num_class = len(np.unique(y_train)),
        tree_method = 'hist',
        random_state = 42,
        n_jobs  = -1
    )

    # ── BASE LEARNER 2: Random Forest ────────────────────────────────
    rf_model = RandomForestClassifier(
        n_estimators = best['rf_n_estimators'],
        max_depth = best['rf_max_depth'],
        max_features = best['rf_max_features'],
        min_samples_split = best['rf_min_samples_split'],
        min_samples_leaf = best['rf_min_samples_leaf'],
        random_state = 42,
        n_jobs= -1
    )

    # ── BASE LEARNER 3: SVM ──────────────────────────────────────────
    svm_model = SVC(
        C = best['svm_C'],
        gamma = best['svm_gamma'],
        kernel = 'rbf',
        probability = True,
        random_state = 42
    )

    # ── BASE LEARNER 4: LightGBM ─────────────────────────────────────
    lgbm_model = lgb.LGBMClassifier(
        n_estimators = best['lgbm_n_estimators'],
        max_depth = best['lgbm_max_depth'],
        learning_rate = best['lgbm_learning_rate'],
        num_leaves = best['lgbm_num_leaves'],
        subsample = best['lgbm_subsample'],
        colsample_bytree = best['lgbm_colsample_bytree'],
        reg_alpha = best['lgbm_alpha'],
        reg_lambda = best['lgbm_lambda'],
        objective = 'multiclass',
        random_state = 42,
        n_jobs = -1,
        verbose = -1
    )

    # ── STACKING ─────────────────────────────────────────────────────
    stacking_final = StackingClassifier(
        estimators=[
            ('xgb', xgb_model),
            ('rf', rf_model),
            ('svm', svm_model),
            ('lgbm', lgbm_model),
        ],
        final_estimator = meta_model,
        cv = 5,
        stack_method = 'predict_proba',
        n_jobs = -1
    )

    # ── PIPELINE FINAL ───────────────────────────────────────────────
    steps_final = [
        ('capping_outliers', IQRCapper(factor=best['iqr_factor'])),
        ('imputacion', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler()),
    ]
    if best_sampler is not None:
        steps_final.append(('sampler', best_sampler))
    steps_final.append(('classifier', stacking_final))

    pipeline_final = Pipeline(steps_final)

    # Sin sample_weight: SVC dentro del stacking no lo propaga correctamente
    pipeline_final.fit(X_train, y_train)

    joblib.dump(pipeline_final, 'modelo_stacking_ganador.pkl')
    print("¡Modelo final entrenado y guardado correctamente como 'modelo_stacking_ganador.pkl'!")

Reconstruyendo y entrenando el modelo final con toda la data...
¡Modelo final entrenado y guardado correctamente como 'modelo_stacking_ganador.pkl'!


In [14]:
if CHARGE_STACKING:
    pipeline_final = joblib.load('modelo_stacking_ganador.pkl')

    X_train = pd.read_csv("../Train_test_split/X_train.csv")
    y_train = pd.read_csv("../Train_test_split/y_train.csv")
    X_test = pd.read_csv("../Train_test_split/X_test.csv")
    y_test = pd.read_csv("../Train_test_split/y_test.csv")

    y_pred = pipeline_final.predict(X_test)
    y_pred_proba = pipeline_final.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba,
                              multi_class='ovr',
                              average='weighted')

    new_result = {
        'Model': ["Stacking Classifier"],
        'Imbalance_Method': [best_sampler_name],
        'Accuracy': [round(accuracy,  4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall,    4)],
        'F1_Score': [round(f1,        4)],
        'ROC_AUC': [round(roc_auc,   4)]
    }

    print(new_result)

{'Model': ['Stacking Classifier'], 'Imbalance_Method': ['Baseline'], 'Accuracy': [0.7639], 'Precision': [0.7624], 'Recall': [0.7639], 'F1_Score': [0.7619], 'ROC_AUC': [0.9078]}
